# Breast Cancer Binary Classification

In [1]:
import pandas as pd

from keras import layers, backend, Sequential
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

from functools import partial

## Load Datasets

In [2]:
X = pd.read_csv("datasets/cancer-features.csv")
y = pd.read_csv("datasets/cancer-target.csv")

display(X)
display(y)

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,1095.0000,0.9053,8589.0,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,0.5435,0.7339,3398.0,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.12380,0.18660,0.2416,186.0000,275.0000,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,0.7456,0.7869,4585.0,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.14440,0.42450,0.4504,243.0000,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,0.4956,1156.0000,3445.0,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,173.00000
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,198.00000,0.10430,0.1809,0.05883,0.7572,0.7813,5438.0,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.13740,205.00000,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,111.00000,0.11590,0.24390,0.13890,0.1726,0.05623,1176.0000,1256.0000,7673.0,158.70,0.010300,0.02891,0.05198,0.02454,0.01114,0.004239,25.45,26.40,166.10,2027.0,141.00000,0.21130,0.4107,0.2216,206.0000,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,144.00000,0.09791,0.1752,0.05533,0.7655,2463.0000,5203.0,99.04,0.005769,0.02423,0.03950,0.01678,0.01898,0.002498,23.69,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,159.0000,0.05648,0.4564,1075.0000,3425.0,48.55,0.005903,0.03731,0.04730,0.01557,0.01318,0.003892,18.98,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,277.00000,0.35140,152.00000,0.2397,0.07016,726.0000,1595.0000,5772.0,86.22,0.006522,0.06158,0.07117,0.01664,0.02324,0.006185,25.74,39.42,184.60,1821.0,165.00000,0.86810,0.9387,265.0000,0.4087,124.00000


,benign
0,0
1,0
2,0
3,0
4,0
...,...
564,0
565,0
566,0
567,0


## Neural Network

### Definition

In [3]:
def get_neural_network(
    units:int,
    activation:str,
    kernel_initializer:str,
    optimizer:str,
    loss:str,
) -> Sequential:
    backend.clear_session()

    model = Sequential()
    model.add(layers.InputLayer(shape=(30,)))
    model.add(layers.Dense(units=units, activation=activation, kernel_initializer=kernel_initializer))
    model.add(layers.Dropout(rate=0.2))
    model.add(layers.Dense(units=units, activation=activation, kernel_initializer=kernel_initializer))
    model.add(layers.Dropout(rate=0.2))
    model.add(layers.Dense(units=1, activation='sigmoid', kernel_initializer=kernel_initializer))

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['binary_accuracy']
    )

    return model

model = KerasClassifier(model=get_neural_network, verbose=0)

### Tuning

In [ ]:
params = {
    'batch_size': [10, 30],
    'epochs': [50, 100],
    'model__units': [8, 16], 
    'model__activation': ['relu', 'tanh'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__optimizer': ['adam', 'sgd'],
    'model__loss': ['binary_crossentropy']
}

tuning_results = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    verbose=2
).fit(X, y) # 56 min

In [ ]:
print(tuning_results.best_params_) # {'batch_size': 10, 'epochs': 100, 'model__activation': 'relu', 'model__kernel_initializer': 'normal', 'model__loss': 'binary_crossentropy', 'model__optimizer': 'adam', 'model__units': 8}
print(tuning_results.best_score_) # 0.9086787765874865

### Best model evaluation

In [7]:
best_model = KerasClassifier(
    model = partial(
        get_neural_network,
        units=8,
        activation='relu',
        kernel_initializer='normal',
        optimizer='adam',
        loss='binary_crossentropy'
    ),
    epochs=100,
    batch_size=10,
    verbose=0
)

results = cross_val_score(
    estimator=best_model,
    X=X, y=y,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [8]:
print(f"Accuracies: {results}")
print(f"Mean: {results.mean()}, Std: {results.std()}")

Accuracies: [0.80701754 0.89473684 0.94736842 0.93859649 0.89380531]
Mean: 0.8963049215960254, Std: 0.04975339303903367


## Final Model

### Fit with All Samples

In [9]:
classificator = KerasClassifier(
    model = partial(
        get_neural_network,
        units=8,
        activation='relu',
        kernel_initializer='normal',
        optimizer='adam',
        loss='binary_crossentropy'
    ),
    epochs=100,
    batch_size=10
).fit(
    X=X,
    y=y
)

Epoch 1/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.4763 - loss: 0.7593  
Epoch 2/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6116 - loss: 0.6551
Epoch 3/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6344 - loss: 0.6244
Epoch 4/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.6608 - loss: 0.5950
Epoch 5/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7100 - loss: 0.5693
Epoch 6/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7030 - loss: 0.5395
Epoch 7/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7417 - loss: 0.4929
Epoch 8/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7329 - loss: 0.5202
Epoch 9/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7610 - loss: 0.4967
Epoch 10/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7891 - loss: 0.4596
Epoch 11/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7891

### Save the Model

In [10]:
classificator.model_.save('models/classificator.keras')